# PosgenPy Full Analysis Example (Python-only)

This is a python-only version of `PosgenPy_Full_Analysis_Example.ipynb` (the
posgen-driven notebook, at the `v1.1.0` tag). It does
**not** use posgen or libatomprobe at all - the cluster search (DBSCAN + envelope +
erosion, matching posgen's "maximum separation" algorithm), the relabelling/
randomisation control runs, and all the corrections/metrics are done by the
`posgenpy` package in this repo, using only numpy/pandas/scikit-learn/scipy.

It follows the same overall workflow as the original notebook:
1. Order (kNN) sweep, at fixed dmax - the n_min each Order needs, then what it
   finds: composition, size and nearest-neighbour distance distributions, and
   any cluster in 3D at every Order
2. dmax sweep, at the Order chosen in step 1 - the same, across dmax
3. Pick final (dmax, order, n_min)
4. Final cluster search + corrections (edge-cluster removal, Fm deconvolution, metrics)
5. Composition/size-distribution/3D plots
6. CV01 parameter-sensitivity check
7. Save a one-row summary CSV

**Making an existing notebook fast.** The sweeps are where the time goes -
`n_params` values times `n_random + 1` datasets full cluster searches, hours at
the 5-10 values and 20-50 control datasets a real study needs. Two changes fix
that, and nothing else in a notebook has to move:

| was | now |
|---|---|
| `pgp.run_parameter_sweep(...)` | `pgp.run_fast_parameter_sweep(...)` - same arguments, same manifest, same files |
| — | add `n_jobs=-1` to run control datasets in parallel |

On a 1.85M-atom tip that takes 10 dmax values x 51 datasets from ~27 min to 8 s.
The manifests it returns feed `optimise_n_min` and every `plot_*` function
unchanged. Envelope and erosion are skipped for the sweep (they cannot change
what it reads) and come back for the final run at the chosen parameters, in
step 4 - see the Order sweep section for the detail.

**Finding a cluster again.** Every distribution plot here is hoverable and each
point carries its own `cluster_id`. Each sweep section ends with a cell that
takes an id read off its graphs, shows that cluster in 3D, and then shows the
same piece of the tip at every other value of the swept parameter - so a
cluster that looks too big at one Order can be watched splitting at the next.

It runs on the small bundled test dataset (`examples/posgenpyTest_easy.pos`,
generated from `examples/generateTestPos_easy.xml` on the first run, +
`examples/posgenpyTest.rrng`) so it can be run top-to-bottom as-is; swap the
file paths/ion lists for your own data and set `generator_script = None`.

In [ ]:
######################### INPUT REQUIRED #########################
from pathlib import Path

import posgenpy as pgp
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sample_name = "PosgenPyTest"
pos_file_path = "examples/posgenpyTest_easy.pos"
range_file_path = "examples/posgenpyTest.rrng"
output_folder = "pygenpy_output_grid/"

# the test cube is generated from this posgen script on the first run (swap
# both "easy"s for "hard" to get 6% Cu+Ni in the matrix instead of 1%);
# set it to None for your own data
generator_script = "examples/generateTestPos_easy.xml"

core_ions = ["Cu", "Ni"]
bulk_ions = ["Fe"]
# ...or for an RPV steel, for example:
# core_ions = ["Cu", "Ni", "Si", "Mn"]
# bulk_ions = ["Fe", "Mo", "C", "Cr", "P"]

random_runs = 20                 # randomised control datasets per parameter cell
n_jobs = -1                      # control datasets to process at once (-1 = all cores)
confidence_threshold = 0.95
detector_efficiency = 1.0        # synthetic: nothing lost. LEAP5000+ ~0.52, LEAP3000 ~0.37
atomic_density_per_nm3 = 30.0    # the test cube's density. BCC iron 85.49


## Load data

In [ ]:
# the test cube is made on the first run (a few seconds) and reused after
if generator_script and not Path(pos_file_path).exists():
    print(f"generating {pos_file_path} from {generator_script}")
    pgp.generate_synthetic_pos(generator_script, pos_file_path)

pos_df = pgp.read_pos_to_df(pos_file_path)
rrng_df = pgp.read_rrng(range_file_path)

# AP Suite exports don't use the "Range..." line format read_rrng expects -
# fall back to the AP-Suite reader if nothing was parsed.
if rrng_df.empty:
    rrng_df = pgp.read_apt_range_file(range_file_path)

ion_colors = pgp.get_ion_colours_from_rrng(rrng_df)

print(f"{len(pos_df):,} atoms")
rrng_df


## Order (kNN) sweep

Sweep `order` at a fixed `dmax`, running a real cluster search plus
`random_runs` relabelled (mc-shuffled) control searches at each value.

**This is the only line that changes to make the old workflow fast.**
`run_fast_parameter_sweep` takes the same arguments as `run_parameter_sweep`,
returns the same manifest and writes the same files, so everything downstream -
`optimise_n_min`, both real-vs-random graphs, the composition graphs - is
untouched.

Why it is faster: per dataset it computes the distance from every core-species
atom to its k-th nearest core-species neighbour (one kNN query at
`k = max(order)`) and every pair of core-species atoms closer than `max(dmax)`
(one radius query). An atom is a core point at `(order, dmax)` exactly when its
order-th neighbour distance is `< dmax`, and two core points link exactly when
their pair distance is `< dmax`, so every swept value after the first is a
re-threshold of arrays already in memory rather than another walk of the tree.
Cost scales with the number of *datasets*, not datasets x parameter values.

It also skips the envelope and erosion. That is not a loss here: this stage
reads posgen's `Cluster Size(core)` (core+border atoms), and the envelope only
ever adds bulk-tree atoms while erosion only removes them, so neither can change
it. They come back for the final run further down, where composition and volume
do depend on them. The one thing to know is that `size` in these per-cluster
files counts core+border atoms rather than every atom, and only core ions get a
column - which changes what the *absolute* composition graph is normalised by.


In [ ]:
######################### INPUT REQUIRED #########################
order_values = [3, 4, 5, 6, 7, 8]
fixed_dmax = 0.5

order_manifest = pgp.run_fast_parameter_sweep(   # was pgp.run_parameter_sweep
    pos_df, rrng_df, core_ions, bulk_ions,
    swept_param="order", swept_values=order_values,
    fixed_dmax=fixed_dmax, n_min=2, n_relabel_runs=random_runs,
    output_dir=output_folder, prefix=sample_name,
    n_jobs=n_jobs,                               # new: control runs in parallel
    verbose=False,
)


### Plotting real-vs-random cluster counts

In [ ]:
n_min_values = list(range(2, 21, 2))

pgp.plot_real_clusters_across_swept_param(order_manifest, "Order", n_min_values)
pgp.plot_real_cluster_ratio_across_swept_param(
    order_manifest, "Order", n_min_values, threshold=confidence_threshold
)


### Find optimal N_min per Order value

In [ ]:
optimal_n_min_order, real_clusters_order, random_clusters_order = pgp.optimise_n_min(
    order_manifest, n_min_values=range(1, 200), confidence_threshold=confidence_threshold
)
optimal_n_min_order


### Cluster composition vs Order

Read on the core-only sweep results, so the denominator is the core ions -
there is no enveloped bulk in these clusters yet. That makes the "absolute"
graph here a *core-ion* composition, the same quantity as the relative one; the
true absolute composition (core ions as a share of every atom in the cluster,
matrix included) comes from the final full-pipeline run further down.


In [ ]:
pgp.plot_cluster_composition_across_swept_param_absolute(
    order_manifest, core_ions, optimal_n_min_order, "Order", ion_colors=ion_colors
)
pgp.plot_cluster_composition_across_swept_param_relative(
    order_manifest, "Order", core_ions, optimal_n_min_order, ion_colors=ion_colors
)


### Cluster size and nearest-neighbour distance vs Order

The graphs above count clusters. These show the clusters themselves, one point
each, at every Order:

- **Size, real vs randomised** - every cluster the sweep found, before any
  n_min, with each Order's optimal n_min drawn as a black tick across its
  violin. The real clusters above the tick are what that Order keeps; the
  randomised ones show how big chance alone makes a cluster.
- **Nearest-neighbour distance** - for the real clusters that survive each
  Order's own n_min, the centre-to-centre distance to the nearest other
  cluster *at that same Order*. The second graph divides it by the two
  clusters' equivalent radii: **below 1 their equivalent spheres overlap** -
  one precipitate reported as two, or two about to merge into one.

Hover a point for its `cluster_id` and `order` - the two numbers the cell after
these takes to show it in 3D. An id belongs to one Order: the same precipitate
is generally numbered differently at every value.

In [ ]:
# every cluster, real and randomised, before n_min - each Order's n_min marked
order_sizes = pgp.load_sweep_size_table(order_manifest)

pgp.plot_size_distribution(
    order_sizes, x="order", y="size_core", n_min=optimal_n_min_order,
    title=f"{sample_name}: cluster size vs Order (dmax {fixed_dmax})",
).show()

In [ ]:
# the real clusters that survive each Order's own n_min, each with its nearest
# neighbour among the clusters of that same Order
order_clusters = pgp.load_grid_cluster_table(order_manifest, n_min=optimal_n_min_order)

pgp.plot_size_distribution(
    order_clusters, x="order", y="closest_cluster_distance", color=None,
    title=f"{sample_name}: nearest-neighbour cluster distance vs Order [nm]",
).show()

pgp.plot_size_distribution(
    order_clusters, x="order", y="closest_cluster_distance_norm", color=None,
    reference_line=(1, "equivalent spheres touch"),
    title=f"{sample_name}: nearest-neighbour distance / (r₁ + r₂) vs Order",
).show()


### Individual clusters across Order

Pick an Order and one or more `cluster_id`s read off the graphs above at that
Order. The first 3D view is those clusters atom by atom. The second takes the
same piece of the tip at **every** Order, each clipped at its own n_min, one
subplot per Order - turning or zooming one turns them all. A cluster that
splits as the Order rises is one blob in one subplot and two in the next; one
that falls below n_min disappears.

The table lists the chosen Order's biggest clusters and their nearest
neighbours, for picking ids without hovering. Keep a selection to clusters
that sit close together: the comparison frames their joint bounding box, and
two clusters on opposite sides of the tip frame most of the tip.

The sweep never ran the envelope, so these are core+border atoms only - the
matrix shell comes back in the final run.

In [ ]:
######################### INPUT REQUIRED #########################
inspect_order = 5          # one of order_values
order_cluster_ids = None   # cluster_ids read off the graphs above, at that Order;
                           # None = the closest pair there, by distance / (r1 + r2)

order_entry = next(e for e in order_manifest if e["swept_value"] == inspect_order)
at_order = order_clusters.loc[order_clusters["order"] == inspect_order]

if order_cluster_ids is None:
    closest = at_order.nsmallest(1, "closest_cluster_distance_norm").iloc[0]
    order_cluster_ids = [int(closest["cluster_id"]), int(closest["closest_cluster_id_norm"])]

# the biggest clusters at this Order, and their nearest neighbours
display(at_order.nlargest(10, "size_core").set_index("cluster_id")[[
    "size_core", *[ion for ion in core_ions if ion in at_order.columns],
    "closest_cluster_id", "closest_cluster_distance", "closest_cluster_distance_norm",
]].round(2))

print(f"Order {inspect_order}: showing clusters {order_cluster_ids}")
_ = pgp.show_clusters_in_pos(
    order_entry["real_clustered_atoms_path"], order_cluster_ids, ion_colors=ion_colors
)

pgp.compare_selected_clusters_across_param(
    order_manifest, "order", chosen_value=inspect_order,
    selected_clusters=order_cluster_ids, n_min=optimal_n_min_order,
    spread=True, ion_colors=ion_colors,
).show()

## dmax sweep

Same again, sweeping `dmax` at a fixed `order` - the Order the section above
settled on - and the same one-word change.

Runs that both sweeps have in common are computed once: the cell at
`(fixed_order, fixed_dmax)` was already written by the sweep above, and is
picked up from disk rather than repeated.

In [ ]:
######################### INPUT REQUIRED #########################
dmax_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
fixed_order = 5            # the Order chosen from the section above

dmax_manifest = pgp.run_fast_parameter_sweep(   # was pgp.run_parameter_sweep
    pos_df, rrng_df, core_ions, bulk_ions,
    swept_param="dmax", swept_values=dmax_values,
    fixed_order=fixed_order, n_min=2, n_relabel_runs=random_runs,
    output_dir=output_folder, prefix=sample_name,
    n_jobs=n_jobs,
    verbose=False,
)

In [ ]:
pgp.plot_real_clusters_across_swept_param(dmax_manifest, "dmax", n_min_values)
pgp.plot_real_cluster_ratio_across_swept_param(
    dmax_manifest, "dmax", n_min_values, threshold=confidence_threshold
)


In [ ]:
optimal_n_min_dmax, real_clusters_dmax, random_clusters_dmax = pgp.optimise_n_min(
    dmax_manifest, n_min_values=range(1, 200), confidence_threshold=confidence_threshold
)
optimal_n_min_dmax


In [ ]:
pgp.plot_cluster_composition_across_swept_param_absolute(
    dmax_manifest, core_ions, optimal_n_min_dmax, "dmax", ion_colors=ion_colors
)
pgp.plot_cluster_composition_across_swept_param_relative(
    dmax_manifest, "dmax", core_ions, optimal_n_min_dmax, ion_colors=ion_colors
)


### Cluster size and nearest-neighbour distance vs dmax

The same two views as for Order, along dmax. Raising dmax adds links, so along
this axis clusters grow and merge rather than split. Watch the upper tail of
the size distribution climb, and the lower tail of the normalised distance: a
dmax where it drops below 1 is either still breaking single precipitates into
fragments (at the low end) or has started merging neighbouring ones (at the
high end).

In [ ]:
# every cluster, real and randomised, before n_min - each dmax's n_min marked
dmax_sizes = pgp.load_sweep_size_table(dmax_manifest)

pgp.plot_size_distribution(
    dmax_sizes, x="dmax", y="size_core", n_min=optimal_n_min_dmax,
    title=f"{sample_name}: cluster size vs dmax (Order {fixed_order})",
).show()

In [ ]:
# the real clusters that survive each dmax's own n_min, each with its nearest
# neighbour among the clusters of that same dmax
dmax_clusters = pgp.load_grid_cluster_table(dmax_manifest, n_min=optimal_n_min_dmax)

pgp.plot_size_distribution(
    dmax_clusters, x="dmax", y="closest_cluster_distance", color=None,
    title=f"{sample_name}: nearest-neighbour cluster distance vs dmax [nm]",
).show()

pgp.plot_size_distribution(
    dmax_clusters, x="dmax", y="closest_cluster_distance_norm", color=None,
    reference_line=(1, "equivalent spheres touch"),
    title=f"{sample_name}: nearest-neighbour distance / (r₁ + r₂) vs dmax",
).show()

### Individual clusters across dmax

As for Order: pick a dmax and ids off the graphs above. Along this axis the
thing to look for is a merge - two clusters in one subplot that are one blob in
the next - or, at a small dmax, one precipitate still in pieces.

In [ ]:
######################### INPUT REQUIRED #########################
inspect_dmax = 0.5         # one of dmax_values
dmax_cluster_ids = None    # cluster_ids read off the graphs above, at that dmax;
                           # None = the closest pair there, by distance / (r1 + r2)

dmax_entry = next(e for e in dmax_manifest if np.isclose(e["swept_value"], inspect_dmax))
at_dmax = dmax_clusters.loc[np.isclose(dmax_clusters["dmax"], inspect_dmax)]

if dmax_cluster_ids is None:
    closest = at_dmax.nsmallest(1, "closest_cluster_distance_norm").iloc[0]
    dmax_cluster_ids = [int(closest["cluster_id"]), int(closest["closest_cluster_id_norm"])]

# the biggest clusters at this dmax, and their nearest neighbours
display(at_dmax.nlargest(10, "size_core").set_index("cluster_id")[[
    "size_core", *[ion for ion in core_ions if ion in at_dmax.columns],
    "closest_cluster_id", "closest_cluster_distance", "closest_cluster_distance_norm",
]].round(2))

print(f"dmax {inspect_dmax}: showing clusters {dmax_cluster_ids}")
_ = pgp.show_clusters_in_pos(
    dmax_entry["real_clustered_atoms_path"], dmax_cluster_ids, ion_colors=ion_colors
)

pgp.compare_selected_clusters_across_param(
    dmax_manifest, "dmax", chosen_value=inspect_dmax,
    selected_clusters=dmax_cluster_ids, n_min=optimal_n_min_dmax,
    spread=True, ion_colors=ion_colors,
).show()

## Select final parameters

Pick `(dmax, order, n_min)` from the sweep results above, and set the two
distances the sweep did not need: `dbulk`, how far a cluster reaches into the
matrix to capture bulk atoms, and `derode`, which strips enveloped atoms that
are also close to unclustered matrix. The old posgen workflow set both to
`dmax/2`.


In [ ]:
######################### INPUT REQUIRED #########################
selected_dmax = 0.5
selected_order = fixed_order     # the n_min below is the dmax sweep's, which ran at this Order

# the optimal n_min at that dmax, off the dmax sweep above (its manifest runs
# in ascending dmax, whatever order dmax_values was typed in)
selected_n_min = next(
    n for e, n in zip(dmax_manifest, optimal_n_min_dmax)
    if np.isclose(e["swept_value"], selected_dmax)
)

# posgen's dbulk / derode - not used by the sweep, but they decide the final
# clusters' composition and volume
envelope_radius = selected_dmax / 2      # dbulk
d_erosion = selected_dmax / 2            # derode

# excluded from the "cluster size" column, as the matrix element always was
elements_to_exclude = ["Fe"]

print(f"dmax={selected_dmax}, order={selected_order}, n_min={selected_n_min}, "
      f"dbulk={envelope_radius}, derode={d_erosion}")

## Final cluster search: the full pipeline

**This is where the envelope and erosion come back.** The sweep skipped them
because it only reads core+border counts, which they cannot change. Everything
below - composition, volume, volume fraction - depends on exactly which bulk
atoms the envelope wraps into each cluster and which of those erosion strips off
again, so the final run goes through `run_clustering` in full:

1. **core classification + linking** - the DBSCAN/kNN stage the sweep did, at
   the chosen `dmax` and `order`;
2. **size clip** - `N_min` applied to core+border counts *before* the envelope,
   which is what posgen does: a discarded cluster's surrounding bulk atoms stay
   untagged and can still erode its neighbours;
3. **envelope** - bulk-species atoms within `dbulk` of a classified core point
   join the cluster;
4. **erosion** - enveloped atoms within `derode` of untagged matrix are given
   back, trimming the shell down to what is really inside.

No relabelling here. The randomised control runs existed to pick `n_min`, and
that is already decided; running them again would only re-derive a number the
sweep has. `verbose=True` prints what each stage added and removed.

Then the corrections, in the order the old workflow applied them: **Fm
deconvolution first** (before anything is derived from the counts), then the
size columns, nearest-neighbour distances, and edge-cluster detection.


In [ ]:
ion_formula_map = pgp.build_ion_formula_map(rrng_df)
labelled_atoms = pgp.assign_rrng_to_pos(pos_df, rrng_df)


def final_cluster_analysis(dmax, order, n_min, dbulk, derode, print_log=False):
    """One full cluster search + the corrections pipeline, at one parameter set.

    Used for the main run and again for each CV01 perturbation below, so the
    sensitivity check measures the parameters changing and nothing else.
    """

    # --- the search: core + link + size clip + envelope + erosion ---
    atoms = pgp.run_clustering(
        labelled_atoms, core_ions, bulk_ions,
        Dmax=dmax, order=order, N_min=n_min,
        envelope_radius=dbulk, d_erosion=derode,
        verbose=print_log,
    )

    stats = pgp.analyse_clusters(
        atoms, ion_formula_map=ion_formula_map, core_species=core_ions,
        atomic_volume=1 / atomic_density_per_nm3,
    )
    unclustered = pgp.summarise_unclustered(atoms, ion_formula_map=ion_formula_map)

    # --- Fm deconvolution, before anything is derived from the counts ---
    # The 29 Da peak is an Fe/Ni overlap; split it by natural Ni abundance.
    if {"Fm", "Ni", "Fe"} <= set(stats.columns) and stats["Fm"].sum() > 0:
        if print_log:
            print(f"\nDeconvoluting {int(stats['Fm'].sum())} Fm atoms in clusters")
        stats = pgp.deconvolute_Fm_clusters(stats)
        if {"Fm", "Ni", "Fe"} <= set(unclustered.columns):
            unclustered = pgp.deconvolute_Fm_clusters(unclustered)

    # --- size columns, as the old workflow defined them ---
    stats["cluster_size"] = pgp.calculate_cluster_size(
        stats, core_ions, bulk_ions, elements_to_exclude
    )
    stats["cluster_size_with_matrix"] = pgp.calculate_cluster_size(
        stats, core_ions, bulk_ions, []
    )

    # --- nearest-neighbour cluster distance, then edge clusters ---
    stats = pgp.find_knn_cluster_distance(stats)
    stats = pgp.remove_edge_clusters(
        atoms.loc[atoms["ion"].notna()], atoms, stats,
        random_state=0, print_log=print_log,
    )

    return atoms, stats.reset_index(drop=True), unclustered


clustered_atoms, cluster_stats, unclustered_stats = final_cluster_analysis(
    selected_dmax, selected_order, selected_n_min,
    envelope_radius, d_erosion, print_log=True,
)

cluster_stats.to_csv(f"{output_folder}{sample_name}_cluster-stats_corrected.csv", index=False)
unclustered_stats.to_csv(f"{output_folder}{sample_name}_unclustered-stats_corrected.csv", index=False)

print(f"\n{len(cluster_stats)} clusters, "
      f"{int(cluster_stats['is_edge_cluster'].sum())} touching the tip edge")
cluster_stats.head()


## Composition plots (ternary + stacked bar)

In [ ]:
ternary_ions = [ion for ion in core_ions + bulk_ions if ion in cluster_stats.columns][:3]

# `pgp.hover_columns` builds the hover_data spec for any plotly figure: every
# column, cluster_id first, floats to 4 significant figures.
labelled_stats = cluster_stats.assign(
    cluster="Cluster " + cluster_stats["cluster_id"].astype(str)
)

fig = px.scatter_ternary(
    labelled_stats, a=ternary_ions[0], b=ternary_ions[1], c=ternary_ions[2],
    size="size", color="size",
    hover_name="cluster",
    hover_data=pgp.hover_columns(labelled_stats, skip=["cluster"]),
    title=f"{sample_name}: cluster composition (ternary)",
)
fig.show()


In [ ]:
bar_df = cluster_stats.sort_values("size")

fig = go.Figure()
for ion in ternary_ions:
    fig.add_trace(go.Bar(
        x=bar_df["cluster_id"].astype(str), y=bar_df[ion], name=ion,
        customdata=bar_df[["cluster_id", "size", "size_core"]],
        hovertemplate=(
            "<b>Cluster %{customdata[0]}</b><br>"
            f"{ion}: " "%{y}<br>size: %{customdata[1]}"
            "<br>size (core+border): %{customdata[2]}<extra></extra>"
        ),
        marker_color=ion_colors.get(ion),
    ))
fig.update_layout(
    barmode="stack", title=f"{sample_name}: per-cluster composition",
    xaxis_title="Cluster ID", yaxis_title="Atoms", template="simple_white",
)
fig.show()


## The final run's clusters, and finding one in 3D

The first plot is the final (full-pipeline) run's size distribution, one point
per cluster - `size` here counts every atom including the enveloped bulk, and
the tooltip carries the cluster's id, composition, shape metrics and whether it
was flagged as an edge cluster.

Pick an id off that plot and put it in `clusters_to_plot` below to see that
exact cluster atom by atom. The 3D view's tooltips carry `cluster_id` too, so
the trip works in both directions.


In [ ]:
pgp.plot_size_distribution(
    cluster_stats, x=None, y="size", color=None,
    title=f"{sample_name}: final cluster sizes (hover for the cluster id)",
).show()

######################### INPUT REQUIRED #########################
# any cluster_id read off the plot above
clusters_to_plot = cluster_stats.sort_values("size", ascending=False)["cluster_id"].head(3).tolist()

print("showing clusters:", clusters_to_plot)
pgp.show_clusters_in_pos(clustered_atoms, clusters_to_plot, ion_colors=ion_colors)

## CV01 sensitivity

Perturb the selected `(dmax, order)` by `(+-0.05 nm, +-1)` and re-run the same
full pipeline twice more, recording the largest absolute deviation per metric as
an error estimate.

It calls `final_cluster_analysis` again, so the perturbed runs go through the
identical envelope, erosion, Fm deconvolution and edge-cluster steps - the only
thing that differs is the two parameters. `dbulk`/`derode` track the perturbed
dmax, keeping the `dmax/2` convention.


In [ ]:
######################### INPUT REQUIRED #########################
dmax_interval = 0.05
order_interval = 1

def metrics_for(name, stats, unclustered, dmax, order, analysis_type):
    return pgp.get_cluster_metrics(
        reconstruction_name=name, reconstruction_location=output_folder,
        sample_type=sample_name, cluster_stats_df=stats,
        unclustered_stats_df=unclustered,
        core_ions=core_ions, bulk_ions=bulk_ions, dmax=dmax, order=order,
        n_min=selected_n_min, detector_efficiency=detector_efficiency,
        atomic_density_per_nm3=atomic_density_per_nm3,
        excluded_Fe="Fe" in elements_to_exclude,
        cluster_analysis_type=analysis_type,
    )

main_metrics = metrics_for(
    sample_name, cluster_stats, unclustered_stats,
    selected_dmax, selected_order, "main",
)

cv01_metrics = []
for sign in (-1, 1):
    this_dmax = round(selected_dmax + sign * dmax_interval, 2)
    this_order = selected_order + sign * order_interval

    _, cv_stats, cv_unclustered = final_cluster_analysis(
        this_dmax, this_order, selected_n_min,
        this_dmax / 2, this_dmax / 2,
    )
    cv01_metrics.append(metrics_for(
        f"{sample_name}_cv01_{sign}", cv_stats, cv_unclustered,
        this_dmax, this_order, "cv01",
    ))
    print(f"cv01 dmax={this_dmax} order={this_order}: {len(cv_stats)} clusters")

cluster_analysis_summary = pd.DataFrame([main_metrics])
cv01_df = pd.DataFrame(cv01_metrics)

cols_for_cv01 = [
    "volume fraction", "number density per nm3", "clustered volume nm3",
    "number of all clusters", "number of non-edge clusters", "number of edge clusters",
]
for col in cols_for_cv01:
    cluster_analysis_summary[f"{col} cv01 error"] = (
        cv01_df[col] - main_metrics[col]
    ).abs().max()

cluster_analysis_summary.T


## Save summary + light disk cleanup

In [ ]:
cluster_analysis_summary.to_csv(
    f"{output_folder}{sample_name}_cluster_analysis_summary.csv", index=False
)

# The final run's atoms only exist in memory (no relabelling, so nothing was
# written for it) - persist them so a cluster can be looked up by id later
# without re-running the search.
clustered_atoms.to_parquet(
    f"{output_folder}{sample_name}_final_clustered_atoms.parquet", index=False
)

# Unlike the posgen-driven workflow there is no per-relabel .pos file explosion
# to prune - only the sweeps' per-value real_clustered_atoms.parquet files are
# worth dropping, and only once you have stopped looking clusters up by id in
# the 3D views above.
import glob, os

sweep_atom_files = glob.glob(
    f"{output_folder}{sample_name}_order_*_dmax_*_real_clustered_atoms.parquet"
)
for path in sweep_atom_files:
    os.remove(path)

print(f"Removed {len(sweep_atom_files)} per-sweep-value atom files. Done.")


---
Thanks for using PosgenPy!
